# ⚙️ Loan Approval Prediction System
## Step 2: Data Preprocessing & Pipeline Construction

Welcome to the data preprocessing stage of the **Loan Approval Prediction System**.

In this notebook, we take the raw data and prepare it for machine learning algorithms. Machine learning models cannot directly understand raw text categories (`Graduate`, `Not Graduate`, `Approved`, `Rejected`), and numerical columns with vastly different scales (e.g. ₹9,900,000 income vs. 12-year term) can bias model weights.

### Key Preprocessing Principles Covered:
1. **Removing Non-Predictive Features** (`loan_id`)
2. **Handling Suspicious / Invalid Values** (Negative assets) without blind row deletion
3. **Target Variable Encoding** (`Approved` → 1, `Rejected` → 0)
4. **Stratified Train-Test Splitting** (`random_state=42`, `stratify=y`)
5. **Preventing Data Leakage** (Fitting transformers **only** on training data)
6. **Reusable Scikit-Learn `ColumnTransformer` Pipeline**
7. **Exporting Preprocessor** for subsequent model training and deployment

---


In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

print("Libraries imported successfully!")

Libraries imported successfully!


### 1. Load Original Dataset
We load `loan_approval_dataset.csv` from the `../data/` directory and clean whitespace from column headers and categorical string values.

In [2]:
# Path to the dataset
data_path = os.path.join('..', 'data', 'loan_approval_dataset.csv')
df = pd.read_csv(data_path)

# Clean column headers and text values (strip whitespace)
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip()

print(f"Original dataset shape: {df.shape}")
print(f"Total rows: {df.shape[0]}, Total columns: {df.shape[1]}")

Original dataset shape: (4269, 13)
Total rows: 4269, Total columns: 13


### 2. Remove `loan_id` Feature
`loan_id` is an applicant index. It has no statistical relationship with credit risk or loan approval. Leaving arbitrary IDs in the dataset can cause models to overfit on noise.


In [3]:
# Drop loan_id
df = df.drop(columns=['loan_id'])

print(f"Shape after removing loan_id: {df.shape}")

Shape after removing loan_id: (4269, 12)


### 3. Check for Suspicious / Invalid Values
We check the range of numerical columns against business rules:
- CIBIL score must be between 300 and 900.
- Loan terms and incomes must be positive.
- Asset values (residential, commercial, luxury, bank) cannot be negative in physical market value.


In [4]:
num_cols = df.select_dtypes(include=[np.number]).columns

print("Numerical Summary (Min, Mean, Max):")
summary = df[num_cols].describe().T[['min', 'mean', 'max']]
print(summary)

print("\nChecking for negative values:")
for col in num_cols:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f"⚠️ Column '{col}' has {neg_count} negative entries! Min value: {df[col].min()}")

Numerical Summary (Min, Mean, Max):
                               min          mean         max
no_of_dependents               0.0  2.498712e+00         5.0
income_annum              200000.0  5.059124e+06   9900000.0
loan_amount               300000.0  1.513345e+07  39500000.0
loan_term                      2.0  1.090045e+01        20.0
cibil_score                  300.0  5.999361e+02       900.0
residential_assets_value -100000.0  7.472617e+06  29100000.0
commercial_assets_value        0.0  4.973155e+06  19400000.0
luxury_assets_value       300000.0  1.512631e+07  39200000.0
bank_asset_value               0.0  4.976692e+06  14700000.0

Checking for negative values:
⚠️ Column 'residential_assets_value' has 28 negative entries! Min value: -100000


#### 🛠️ Handling Negative Asset Values
We identified **28 records** where `residential_assets_value` is `-100,000`.
Instead of blindly deleting these 28 rows (which causes data loss), we **clip negative values to 0** (`df['residential_assets_value'].clip(lower=0)`).
This represents that the applicant has ₹0 in residential asset equity, preserving all 4,269 records.


In [5]:
# Clip negative residential asset values to 0
negative_before = (df['residential_assets_value'] < 0).sum()
df['residential_assets_value'] = df['residential_assets_value'].clip(lower=0)
negative_after = (df['residential_assets_value'] < 0).sum()

print(f"Negative values before handling: {negative_before}")
print(f"Negative values after clipping : {negative_after}")
print(f"New minimum residential_assets_value: ₹{df['residential_assets_value'].min():,}")

Negative values before handling: 28
Negative values after clipping : 0
New minimum residential_assets_value: ₹0


### 4. Separate Features `X` and Target `y`
We separate the independent predictor features `X` from the dependent target `y` (`loan_status`).


In [6]:
X = df.drop(columns=['loan_status'])
y_raw = df['loan_status']

print(f"Feature matrix (X) shape: {X.shape}")
print(f"Target column (y) name  : loan_status")
print(f"Feature columns ({len(X.columns)}):\n", list(X.columns))

Feature matrix (X) shape: (4269, 11)
Target column (y) name  : loan_status
Feature columns (11):
 ['no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value']


### 5. Encode Target Variable `loan_status`
In binary classification, the outcome of interest (positive class) is mapped to `1` and the negative class to `0`:
- **`Approved`** → **`1`** (Loan Granted)
- **`Rejected`** → **`0`** (Loan Denied)


In [7]:
# Map target values to binary
y = y_raw.map({'Approved': 1, 'Rejected': 0})

print("Encoded target values:")
print(pd.DataFrame({
    'Raw Target': ['Approved', 'Rejected'],
    'Encoded Value': [1, 0]
}))

print("\nTarget Value Counts & Proportions:")
counts = y.value_counts()
pcts = y.value_counts(normalize=True) * 100
target_dist = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts.round(2)})
print(target_dist)

Encoded target values:
  Raw Target  Encoded Value
0   Approved              1
1   Rejected              0

Target Value Counts & Proportions:
             Count  Percentage (%)
loan_status                       
1             2656           62.22
0             1613           37.78


### 6. Stratified Train-Test Split
We split the data into **80% training** and **20% testing** subsets:
- `random_state=42`: Ensures our experiment is 100% reproducible.
- `stratify=y`: Guarantees that both train and test splits have the exact same ratio of Approved (62.2%) vs. Rejected (37.8%) applicants.


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape (X_train): {X_train.shape}")
print(f"Testing set shape  (X_test) : {X_test.shape}")
print(f"Training labels (y_train)   : {y_train.shape}")
print(f"Testing labels  (y_test)    : {y_test.shape}")

print("\nClass balance check (Proportion Approved in Train vs Test):")
print(f"  y_train Approved %: {(y_train.mean()*100):.2f}%")
print(f"  y_test  Approved %: {(y_test.mean()*100):.2f}%")

Training set shape (X_train): (3415, 11)
Testing set shape  (X_test) : (854, 11)
Training labels (y_train)   : (3415,)
Testing labels  (y_test)    : (854,)

Class balance check (Proportion Approved in Train vs Test):
  y_train Approved %: 62.23%
  y_test  Approved %: 62.18%


### 7. Scikit-Learn Preprocessing Pipeline (`ColumnTransformer`)
We construct a reusable `ColumnTransformer`:
1. **Numerical Transformer (`StandardScaler`)**:
   Standardizes numerical features by removing the mean and scaling to unit variance:
   $$z = \frac{x - \mu}{\sigma}$$
2. **Categorical Transformer (`OneHotEncoder`)**:
   Converts categorical text columns (`education`, `self_employed`) into binary indicator columns.
   Using `drop='first'` prevents multicollinearity (dummy variable trap).

> ⚠️ **Data Leakage Prevention:**
> The preprocessor is **fitted strictly on `X_train`** (`preprocessor.fit(X_train)`).
> The test data (`X_test`) is only **transformed** (`preprocessor.transform(X_test)`), ensuring no information from the test set leaks into the preprocessing step.


In [9]:
# Identify numerical and categorical columns
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

# Construct ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_cols)
    ],
    remainder='drop'
)

# Fit ONLY on training data
preprocessor.fit(X_train)

# Transform both train and test data
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("\nPreprocessing pipeline fitted on X_train and applied successfully!")

Numerical columns (9): ['no_of_dependents', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value']
Categorical columns (2): ['education', 'self_employed']

Preprocessing pipeline fitted on X_train and applied successfully!


### 8. Final Processed Feature Information
Let's inspect the final feature names and shape of the transformed matrices.

In [10]:
feature_names = preprocessor.get_feature_names_out().tolist()

print("Final Processed Feature Information:")
print(f"Total processed features: {len(feature_names)}\n")
for idx, name in enumerate(feature_names, 1):
    print(f"  {idx:2d}. {name}")

print(f"\nX_train_processed shape: {X_train_processed.shape}")
print(f"X_test_processed shape : {X_test_processed.shape}")

# Preview first 3 rows of processed training data
df_processed_sample = pd.DataFrame(X_train_processed[:3], columns=feature_names)
print("\nFirst 3 rows of processed training data:")
df_processed_sample

Final Processed Feature Information:
Total processed features: 11

   1. num__no_of_dependents
   2. num__income_annum
   3. num__loan_amount
   4. num__loan_term
   5. num__cibil_score
   6. num__residential_assets_value
   7. num__commercial_assets_value
   8. num__luxury_assets_value
   9. num__bank_asset_value
  10. cat__education_Not Graduate
  11. cat__self_employed_Yes

X_train_processed shape: (3415, 11)
X_test_processed shape : (854, 11)

First 3 rows of processed training data:


,num__no_of_dependents,num__income_annum,num__loan_amount,num__loan_term,num__cibil_score,num__residential_assets_value,num__commercial_assets_value,num__luxury_assets_value,num__bank_asset_value,cat__education_Not Graduate,cat__self_employed_Yes
0,0.890718,-0.092646,-0.088677,-0.850759,1.131102,-0.235274,-1.137306,-0.444440,-0.736827,0.0,0.0
1,0.301107,0.265460,0.180218,1.243433,0.194245,0.107512,0.785995,-0.048387,-0.396795,0.0,1.0
2,-0.288503,1.089102,0.807638,1.243433,0.795684,2.320042,1.747645,1.304793,0.716035,0.0,1.0


### 9. Save Preprocessor for Model Training & Deployment
We save the fitted preprocessor using `joblib` so that later model training and the Flask web application can apply the exact same transformation to incoming applicant data.

In [11]:
models_dir = os.path.join('..', 'models')
os.makedirs(models_dir, exist_ok=True)

preprocessor_path = os.path.join(models_dir, 'preprocessor.joblib')
joblib.dump(preprocessor, preprocessor_path)

print(f"Preprocessor saved successfully to: {preprocessor_path}")
print(f"File size: {os.path.getsize(preprocessor_path):,} bytes")

Preprocessor saved successfully to: ..\models\preprocessor.joblib
File size: 3,875 bytes


---
## 🎯 Preprocessing Summary
1. **Identifier Removed**: `loan_id` dropped.
2. **Invalid Values Handled**: 28 negative entries in `residential_assets_value` clipped to 0 (no data loss).
3. **Target Encoded**: `Approved` → `1`, `Rejected` → `0`.
4. **Stratified Split**:
   - `X_train` shape: `(3415, 11)`
   - `X_test` shape: `(854, 11)`
   - `y_train` shape: `(3415,)`
   - `y_test` shape: `(854,)`
5. **Leakage-Free Pipeline**: `StandardScaler` + `OneHotEncoder(drop='first')` fitted solely on `X_train`.
6. **Artifact Persisted**: Saved to `models/preprocessor.joblib`.
